# Exploración interactiva: alertas, retroalimentación humana y recalibración

Notebook para probar a mano el flujo de HU4-HU6 (modelo predictivo → alertas → retroalimentación humana → recalibración supervisada) mientras no existe todavía una interfaz de usuario (HU6/frontend, ADR-0003). Reemplaza este notebook por la interfaz real cuando esté disponible.

**Requiere `data/melchor_romero_2024_consolidado.parquet`**, que **no está versionado en git** (es un dato local, `.gitignore` según ADR-0002) — hay que generarlo (HU2) o subirlo a mano.

- **Local (VS Code / Jupyter)**: cloná el repo, corré `pip install -e ".[dev]"` desde la raíz, y asegurate de que el archivo exista en `data/`.
- **Google Colab**: la celda siguiente clona el repo en el entorno de Colab (no queda nada en tu máquina local) e instala el paquete automáticamente; si `data/melchor_romero_2024_consolidado.parquet` no existe, te va a pedir que lo subas manualmente (botón de carga de archivo).

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Ejecutando en Colab: {IN_COLAB}")

if IN_COLAB:
    REPO_URL = "https://github.com/gusjrivas/AAI_Hydric_Stress.git"
    REPO_DIR = Path("AAI_Hydric_Stress")

    if not REPO_DIR.exists():
        get_ipython().system(f"git clone -q {REPO_URL}")

    import os

    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")

    dataset_path = Path("data/melchor_romero_2024_consolidado.parquet")
    if not dataset_path.exists():
        from google.colab import files

        print(
            "No se encontró data/melchor_romero_2024_consolidado.parquet "
            "(no está versionado en git). Subilo ahora desde el diálogo de carga:"
        )
        uploaded = files.upload()
        dataset_path.parent.mkdir(parents=True, exist_ok=True)
        for name in uploaded:
            Path(name).replace(dataset_path.parent / name)
else:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from architecture_integration.pipeline import run_end_to_end_pipeline
from data_ingestion.storage import load_dataset
from human_feedback.recalibration import recalibrate_model, select_recalibration_observations
from human_feedback.registry import integrate_feedback_with_predictions
from human_feedback.schema import update_feedback
from predictive_modeling.models import build_candidate_models

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Ejecutar el pipeline completo

Corre calidad + modelado + alertas + inicialización de retroalimentación en un solo paso (mismo orquestador que `scripts/run_end_to_end_pipeline.py`).

In [ ]:
df = load_dataset("melchor_romero_2024_consolidado")
split_date = df["timestamp"].sort_values().iloc[int(len(df) * 0.8)].date()

model = build_candidate_models(random_state=42)["random_forest"]

result = run_end_to_end_pipeline(
    df,
    label_column="soil_moisture",
    feature_columns=["soil_moisture", "solar_radiation", "relative_humidity"],
    split_date=split_date,
    model=model,
    include_anomaly_detection=True,
)

print(f"split_date: {split_date}")
print(f"train: {len(result['train'])} filas, test: {len(result['test'])} filas")
print(f"alertas generadas: {int(result['alerts'].sum())} de {len(result['alerts'])}")

## 2. Ver las alertas generadas y elegir una para validar

Cambiá el índice de `feedback_log` en las celdas siguientes para probar con otras fechas.

In [ ]:
feedback_log = result["feedback_log"]
feedback_log[feedback_log["alerta_generada"] == 1].head(10)

## 3. Confirmar una alerta

Elegí una fecha de la tabla de arriba y marcala como `confirmada` (la alerta era correcta).

In [ ]:
fecha_a_confirmar = feedback_log.loc[feedback_log["alerta_generada"] == 1, "fecha"].iloc[0]

feedback_log = update_feedback(
    feedback_log, fecha=fecha_a_confirmar, estado_validacion="confirmada"
)

feedback_log[feedback_log["fecha"] == fecha_a_confirmar]

## 4. Rechazar una alerta con corrección

Elegí otra fecha (puede ser con o sin alerta) y marcala como `rechazada`, indicando la etiqueta correcta (0 = sin estrés, 1 = con estrés) y una observación.

In [ ]:
fecha_a_rechazar = feedback_log["fecha"].iloc[3]

feedback_log = update_feedback(
    feedback_log,
    fecha=fecha_a_rechazar,
    estado_validacion="rechazada",
    etiqueta_corregida=0,  # cambiar a 1 si corresponde
    observacion="corrección de prueba desde el notebook",
)

feedback_log[feedback_log["fecha"] == fecha_a_rechazar]

## 5. Integrar la retroalimentación con las predicciones

Une, por fecha, el estado de validación con la probabilidad predicha y la etiqueta real.

In [ ]:
predictions = pd.DataFrame(
    {
        "fecha": result["test"]["timestamp"].reset_index(drop=True),
        "y_proba": result["y_proba"],
        "stress_label": result["test"]["stress_label"].reset_index(drop=True),
    }
)

integrated = integrate_feedback_with_predictions(feedback_log, predictions)
integrated.head(10)

## 6. Seleccionar observaciones de recalibración y recalibrar el modelo

Solo las alertas `rechazada` con `etiqueta_corregida` califican. Compará las predicciones del modelo original contra el recalibrado en esas mismas fechas (si están en el conjunto de entrenamiento, verás el cambio directamente; si están en test, recalibrá con más correcciones primero).

In [ ]:
recalibration_obs = select_recalibration_observations(integrated)
print(f"observaciones seleccionadas para recalibración: {len(recalibration_obs)}")
recalibration_obs

In [ ]:
# Nota: la retroalimentación de la sección 4 corrige una alerta del conjunto
# de EVALUACIÓN (donde efectivamente se revisan alertas), pero `recalibrate_model`
# reentrena sobre el conjunto de ENTRENAMIENTO. Esto es intencional: en producción,
# las alertas revisadas hoy pasan a ser historia (datos de entrenamiento) en el
# próximo reentrenamiento, no en el mismo ciclo. Por eso, si `recalibration_obs`
# no tiene fechas en `result["train"]`, esta celda no mostrará ningún cambio.

X_train = result["train"][result["feature_columns"]]
y_train = result["train"]["stress_label"]
dates_train = result["train"]["timestamp"]

overlap = dates_train.isin(recalibration_obs["fecha"]) if len(recalibration_obs) > 0 else pd.Series([], dtype=bool)

if overlap.any():
    recalibrated_model, corrected_y_train = recalibrate_model(
        result["model"], X_train, y_train, dates_train, recalibration_obs
    )
    print("Etiquetas de entrenamiento reemplazadas en:")
    display(corrected_y_train[overlap])
else:
    print("Ninguna observación de recalibración cae en el conjunto de entrenamiento todavía.")
    print("Ver la celda siguiente para una demostración directa del mecanismo.")

## 7. Demostración directa: corregir una fecha del conjunto de entrenamiento

Para ver el efecto de la recalibración de forma inmediata (sin esperar al próximo ciclo de reentrenamiento), esta celda inyecta una corrección directamente sobre una fecha de `train` — igual criterio usado para verificar `recalibrate_model` en HU5 con datos reales.

In [ ]:
fecha_demo = dates_train.iloc[10]
etiqueta_original = int(y_train.iloc[10])
etiqueta_corregida_demo = 1 - etiqueta_original

demo_obs = pd.DataFrame({"fecha": [fecha_demo], "etiqueta_corregida": [etiqueta_corregida_demo]})

recalibrated_model_demo, corrected_y_train_demo = recalibrate_model(
    result["model"], X_train, y_train, dates_train, demo_obs
)

fila = X_train.iloc[[10]]
print(f"fecha: {fecha_demo}, etiqueta original: {etiqueta_original}, corregida: {etiqueta_corregida_demo}")
print(f"predicción modelo original:    {result['model'].predict(fila)[0]}")
print(f"predicción modelo recalibrado: {recalibrated_model_demo.predict(fila)[0]}")